In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(42)
tf.random.set_seed(42)

In [3]:
summary = pd.read_csv('cycle_summary_soh_rul_WITH_IMPEDANCE.csv')
summary = summary.sort_values(['battery_id', 'cycle'])

print("Before backfill - missing Re/Rct:", summary['Re_ohm'].isna().sum(), "of", len(summary))

# Backfill per battery: early cycles (before NASA started interleaving
# impedance measurements) borrow the first available reading from later
# in that same battery's life. Defensible since resistance is roughly
# flat before aging accelerates.
summary['Re_ohm'] = summary.groupby('battery_id')['Re_ohm'].bfill()
summary['Rct_ohm'] = summary.groupby('battery_id')['Rct_ohm'].bfill()

print("After backfill - missing Re/Rct:", summary['Re_ohm'].isna().sum(), "of", len(summary))

# If any battery has ZERO impedance readings at all, bfill can't help --
# check for that case explicitly
still_missing = summary[summary['Re_ohm'].isna()]['battery_id'].unique()
if len(still_missing) > 0:
    print("WARNING: these batteries have NO impedance data at all:", still_missing)
    print("Dropping their rows since there's nothing to fill from.")
    summary = summary.dropna(subset=['Re_ohm', 'Rct_ohm'])

# Lag feature: previous cycle's SOH (legitimate -- you always know your
# own past cycle history on real hardware)
summary['soh_prev'] = summary.groupby('battery_id')['soh'].shift(1)
summary['soh_prev'] = summary['soh_prev'].fillna(1.0)

feature_cols = [
    'cycle', 'max_voltage_V', 'min_voltage_V', 'mean_voltage_V',
    'max_current_A', 'min_current_A', 'mean_current_A',
    'max_temperature_C', 'min_temperature_C', 'mean_temperature_C',
    'discharge_time_s', 'soh_prev',
    'Re_ohm', 'Rct_ohm'   # <-- new impedance features
]
target_col = 'soh'

print("\nFeatures used:", feature_cols)

Before backfill - missing Re/Rct: 1615 of 2524
After backfill - missing Re/Rct: 29 of 2524
 'B0054' 'B0055' 'B0056']
Dropping their rows since there's nothing to fill from.

Features used: ['cycle', 'max_voltage_V', 'min_voltage_V', 'mean_voltage_V', 'max_current_A', 'min_current_A', 'mean_current_A', 'max_temperature_C', 'min_temperature_C', 'mean_temperature_C', 'discharge_time_s', 'soh_prev', 'Re_ohm', 'Rct_ohm']


In [4]:
batteries = sorted(summary['battery_id'].unique())
rng = np.random.RandomState(42)
shuffled = rng.permutation(batteries)
n_test = max(1, int(len(batteries) * 0.2))
test_batteries = set(shuffled[:n_test])
train_batteries = set(shuffled[n_test:])

train_df = summary[summary['battery_id'].isin(train_batteries)]
test_df = summary[summary['battery_id'].isin(test_batteries)]

X_train, y_train = train_df[feature_cols].values, train_df[target_col].values
X_test, y_test = test_df[feature_cols].values, test_df[target_col].values
print(X_train.shape, X_test.shape)

(2155, 14) (340, 14)


In [7]:
print(summary[feature_cols].dtypes)

cycle                   int64
max_voltage_V         float64
min_voltage_V         float64
mean_voltage_V        float64
max_current_A         float64
min_current_A         float64
mean_current_A        float64
max_temperature_C     float64
min_temperature_C     float64
mean_temperature_C    float64
discharge_time_s      float64
soh_prev              float64
Re_ohm                 object
Rct_ohm                object
dtype: object


In [5]:
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
std[std == 0] = 1.0
X_train_n = (X_train - mean) / std
X_test_n = (X_test - mean) / std

TypeError: unsupported operand type(s) for /: 'str' and 'int'

In [6]:
lr = LinearRegression()
lr.fit(X_train_n, y_train)
lr_pred = lr.predict(X_test_n)
lr_mae = mean_absolute_error(y_test, lr_pred)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))
lr_r2 = r2_score(y_test, lr_pred)
print(f'Linear Regression -> MAE: {lr_mae:.4f}  RMSE: {lr_rmse:.4f}  R2: {lr_r2:.4f}')

NameError: name 'X_train_n' is not defined

In [ ]:
rf = RandomForestRegressor(n_estimators=200, max_depth=10, n_jobs=-1, random_state=42)
rf.fit(X_train_n, y_train)
rf_pred = rf.predict(X_test_n)
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)
print(f'Random Forest -> MAE: {rf_mae:.4f}  RMSE: {rf_rmse:.4f}  R2: {rf_r2:.4f}')

In [ ]:
dnn = keras.Sequential([
    keras.layers.Input(shape=(X_train_n.shape[1],)),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dropout(0.15),
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dropout(0.15),
    keras.layers.Dense(8, activation='relu'),
    keras.layers.Dense(1)
])
dnn.compile(optimizer='adam', loss='mse')

early_stop = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)

history = dnn.fit(
    X_train_n, y_train,
    validation_split=0.15,
    epochs=100, batch_size=32, verbose=1,
    callbacks=[early_stop]
)

dnn_pred = dnn.predict(X_test_n, verbose=0).flatten()
dnn_mae = mean_absolute_error(y_test, dnn_pred)
dnn_rmse = np.sqrt(mean_squared_error(y_test, dnn_pred))
dnn_r2 = r2_score(y_test, dnn_pred)
print(f'DNN -> MAE: {dnn_mae:.4f}  RMSE: {dnn_rmse:.4f}  R2: {dnn_r2:.4f}')
print(f'Stopped at epoch {len(history.history["loss"])}')

In [ ]:
test_df_check = test_df.copy()
test_df_check['rf_pred'] = rf_pred

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, pred) in zip(axes, [('Linear Regression', lr_pred), ('Random Forest', rf_pred), ('DNN', dnn_pred)]):
    ax.scatter(y_test, pred, s=5, alpha=0.4)
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
    ax.set_xlabel('True SOH'); ax.set_ylabel('Predicted SOH')
    ax.set_title(name)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8,5))
for b in test_df_check['battery_id'].unique():
    sub = test_df_check[test_df_check['battery_id'] == b]
    plt.scatter(sub['soh'], sub['rf_pred'], s=15, alpha=0.6, label=b)
plt.plot([0.3, 1.15], [0.3, 1.15], 'r--')
plt.xlabel('True SOH'); plt.ylabel('Predicted SOH (RF)')
plt.legend(markerscale=1.5)
plt.title('RF Predictions by Battery (with impedance features)')
plt.show()